# Telco Customer Churn

Projet de préparation et nettoyage de données.

L'objectif est de transformer un dataset brut en un jeu de données exploitable pour des algorithmes de Machine Learning.

## Phase 0 : Chargement du dataset

Avant toute opération de nettoyage, il est nécessaire d'examiner la structure générale du jeu de données.

Cette première étape permet de vérifier que les données sont correctement chargées et d'obtenir une première vue des variables disponibles.

In [4]:
import pandas as pd

In [8]:
df = pd.read_csv("../WA_Fn-UseC_-Telco-Customer-Churn.csv")
print("Forme :", df.shape)
print(df.dtypes)
df.head()

Forme : (7043, 21)
customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Phase 1 : Audit qualité

Avant toute opération de nettoyage, il est nécessaire d'établir un état des lieux du dataset.

L'objectif est d'identifier les éventuels problèmes de qualité des données et d'observer la répartition de la variable cible `Churn`.

In [9]:
def audit_qualite(df):

    print(f"Forme : {df.shape}")

    print("\nTypes des colonnes :")
    print(df.dtypes)

    taux_manquants = (
        df.isna()
        .mean()
        .sort_values(ascending=False)
        * 100
    )

    print("\nPourcentage de valeurs manquantes :")
    print(taux_manquants[taux_manquants > 0])

    repartition = df["Churn"].value_counts()
    pourcentage = df["Churn"].value_counts(normalize=True) * 100

    print("\nRépartition de Churn :")

    for valeur in repartition.index:
        print(
            f"Churn {valeur} : "
            f"{repartition[valeur]} "
            f"({pourcentage[valeur]:.1f}%)"
        )

In [10]:
audit_qualite(df)

Forme : (7043, 21)

Types des colonnes :
customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object

Pourcentage de valeurs manquantes :
Series([], dtype: float64)

Répartition de Churn :
Churn No : 5174 (73.5%)
Churn Yes : 1869 (26.5%)


### Première lecture

Ce rapport permet d'identifier rapidement les caractéristiques générales du dataset ainsi que l'équilibre de la variable cible.

La répartition de `Churn` sera particulièrement importante lors des futures phases de modélisation.

### Observation

Une anomalie apparaît dans les types de données : la variable `TotalCharges` est interprétée comme du texte (`str`) alors qu'elle représente un montant.

Cela suggère la présence de valeurs non numériques ou de données manquantes déguisées, qui devront être investiguées avant toute modélisation.

In [12]:
df["TotalCharges"].str.strip().eq("").sum()

np.int64(11)

### Investigation complémentaire

L'audit n'a révélé aucune valeur manquante au sens de Pandas.

Cependant, la colonne `TotalCharges` est interprétée comme du texte alors qu'elle représente un montant.

Une vérification supplémentaire montre que 11 lignes contiennent une chaîne vide après suppression des espaces.

Ces valeurs constituent probablement des données manquantes déguisées qui devront être traitées avant toute modélisation.

In [13]:
df_churn_yes = df[df["Churn"] == "Yes"]

audit_qualite(df_churn_yes)

Forme : (1869, 21)

Types des colonnes :
customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object

Pourcentage de valeurs manquantes :
Series([], dtype: float64)

Répartition de Churn :
Churn Yes : 1869 (100.0%)


In [14]:
df_desequilibre = pd.concat(
    [
        df[df["Churn"] == "No"],
        df[df["Churn"] == "Yes"].sample(
            50,
            random_state=42
        )
    ]
)

audit_qualite(df_desequilibre)

Forme : (5224, 21)

Types des colonnes :
customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object

Pourcentage de valeurs manquantes :
Series([], dtype: float64)

Répartition de Churn :
Churn No : 5174 (99.0%)
Churn Yes : 50 (1.0%)


### Validation de l'audit

Les différents scénarios testés montrent que la fonction reste stable lorsque la distribution de la cible change fortement.

Le déséquilibre artificiellement créé apparaît immédiatement dans le rapport, ce qui permettra d'anticiper certains biais lors de l'entraînement des modèles.

L'audit a également permis d'identifier une anomalie sur la colonne `TotalCharges`, révélant la présence de valeurs manquantes déguisées qui ne sont pas détectées par les fonctions classiques de Pandas.

## Phase 2 : Réparation de la colonne `TotalCharges`

L'audit précédent a mis en évidence une anomalie sur la colonne `TotalCharges`.

Bien qu'elle représente un montant, Pandas l'interprète comme du texte. Une investigation complémentaire a révélé la présence de valeurs vides dissimulées sous forme d'espaces.

L'objectif de cette phase est de convertir cette colonne en numérique et de traiter les valeurs manquantes ainsi révélées.

In [15]:
def reparer_total_charges(df):

    df = df.copy()

    df["TotalCharges"] = pd.to_numeric(
        df["TotalCharges"],
        errors="coerce"
    )

    nb_nan = df["TotalCharges"].isna().sum()

    print(f"Valeurs manquantes révélées : {nb_nan}")

    mediane = df["TotalCharges"].median()

    df["TotalCharges"] = df["TotalCharges"].fillna(
        mediane
    )

    print(
        f"Type final : {df['TotalCharges'].dtype}"
    )

    return df

In [16]:
df = reparer_total_charges(df)

Valeurs manquantes révélées : 11
Type final : float64


### Choix effectué

La conversion numérique révèle 11 valeurs manquantes qui étaient masquées par des chaînes contenant uniquement des espaces.

Ces valeurs ont été remplacées par la médiane de la colonne.

Ce choix permet de conserver les 7043 observations du dataset tout en limitant l'influence des valeurs extrêmes.

## Phase 3 : Encodage des variables catégorielles

La majorité des algorithmes de Machine Learning nécessitent des données numériques.

Cette étape consiste à transformer les variables textuelles du dataset en variables exploitables par un modèle tout en conservant l'information utile à la prédiction du churn.

In [23]:
def encoder_features(df):

    df = df.copy()

    # Suppression de l'identifiant client
    df = df.drop(columns=["customerID"])

    colonnes_objet = (
        df.select_dtypes(include=["object"])
        .columns
        .tolist()
    )

    colonnes_objet.remove("Churn")

    colonnes_binaires = []

    for colonne in colonnes_objet:

        if df[colonne].nunique() == 2:
            colonnes_binaires.append(colonne)

    # Encodage des colonnes binaires
    for colonne in colonnes_binaires:

        valeurs = sorted(
            df[colonne]
            .dropna()
            .unique()
        )

        mapping = {
            valeurs[0]: 0,
            valeurs[1]: 1
        }

        df[colonne] = df[colonne].map(mapping)

    # Colonnes à plusieurs modalités
    colonnes_nominales = [
        col
        for col in colonnes_objet
        if col not in colonnes_binaires
    ]

    df = pd.get_dummies(
        df,
        columns=colonnes_nominales,
        drop_first=True,
        dtype=int
    )

    # Encodage de la cible
    df["Churn"] = df["Churn"].map({
        "No": 0,
        "Yes": 1
    })

    return df

In [24]:
df_encode = encoder_features(df)

df_encode.head()

C:\Users\guder\AppData\Local\Temp\ipykernel_23460\2088662894.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.select_dtypes(include=["object"])


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,...,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,0,1,0,1,0,1,29.85,29.85,0,...,0,0,0,0,0,0,0,0,1,0
1,1,0,0,0,34,1,0,56.95,1889.50,0,...,0,0,0,0,0,1,0,0,0,1
2,1,0,0,0,2,1,1,53.85,108.15,1,...,0,0,0,0,0,0,0,0,0,1
3,1,0,0,0,45,0,0,42.30,1840.75,0,...,1,0,0,0,0,1,0,0,0,0
4,0,0,0,0,2,1,1,70.70,151.65,1,...,0,0,0,0,0,0,0,0,1,0


In [26]:
print("Forme finale :", df_encode.shape)

print(
    "Colonnes texte restantes :",
    len(
        df_encode.select_dtypes(
            include="object"
        ).columns
    )
)

Forme finale : (7043, 31)
Colonnes texte restantes : 0


In [27]:
df_encode.dtypes.value_counts()

int64      29
float64     2
Name: count, dtype: int64